# 01 — Dataset Exploration & Schema Audit

**Stage 1: NF-UNSW-NB15-v2 Dataset Audit**

All analysis is performed via  — no model code, no preprocessing, no data modification.

## 0. Environment Setup

In [ ]:
import sys
from pathlib import Path

# Resolve project root (works locally and on Kaggle after git clone)
root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"Project root: {root}")

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.audit import (
    audit_file_metadata,
    audit_missing_and_infinite,
    audit_duplicates,
    audit_constant_features,
    audit_label_semantics,
    audit_temporal_provenance,
    audit_expected_features,
    run_full_audit,
    print_audit_summary,
)
from src.utils.config import load_config

config = load_config()
RAW_CSV = root / config["paths"]["raw_data_dir"] / config["paths"]["raw_csv_filename"]
REPORT_PATH = root / config["paths"]["metrics_dir"] / "dataset_audit_report.json"
print(f"Dataset: {RAW_CSV}")
print(f"Report : {REPORT_PATH}")

## 1. Load Pre-Generated Audit Report

The report was generated by  using chunked reads. Load it here for interactive exploration.

In [ ]:
with open(REPORT_PATH) as f:
    report = json.load(f)
print("Report loaded successfully.")

## 2. File-Level Metadata

In [ ]:
fm = report["file_metadata"]
print(f"File size  : {fm['file_size_mb']} MB")
print(f"Total rows : {fm['total_rows']:,}")
print(f"Columns    : {fm['total_columns']}")
print()
print("Column names:")
for i, c in enumerate(fm["column_names"], 1):
    print(f"  {i:2d}. {c}")

## 3. Data Quality

In [ ]:
mi = report["missing_and_infinite"]
dupes = report["duplicates"]
const = report["constant_features"]

print(f"Missing cells  : {mi['total_missing_cells']}")
print(f"Infinite cells : {mi['total_infinite_cells']}")
print(f"Duplicate rows : {dupes['duplicate_rows']:,} ({dupes['duplicate_percentage']}%)")
print(f"Constant feats : {const['num_constant_features']}")
print("
→ Dataset is clean: no missing values, no infinite values, no duplicates, no constant features.")

## 4. Feature Completeness

In [ ]:
fe = report["expected_features"]
print(f"Expected model inputs : 39")
print(f"Observed model inputs : {fe['observed_num_features']}")
print(f"Schema match          : {fe['feature_count_matches']}")
print(f"
Found identifier cols : {fe['found_identifier_columns']}")
print(f"Found label cols      : {fe['found_label_columns']}")
print(f"
Model input features:")
for c in fe["feature_columns"]:
    print(f"  - {c}")

## 5. Label Semantics

In [ ]:
ls = report["label_semantics"]

print(f"Label column unique values : {ls['label_column']['unique_values']}")
print(f"Attack column unique values: {ls['attack_column']['unique_values']}")
print(f"
Inferred binary column     : {ls['inferred_binary_column']}")
print(f"Inferred multiclass column : {ls['inferred_multiclass_column']}")

print("
--- Label distribution ---")
for v, c in sorted(ls["label_column"]["distribution"].items()):
    total = sum(ls["label_column"]["distribution"].values())
    print(f"  Label={v}: {c:>9,}  ({c/total*100:.2f}%)")

print("
--- Attack category distribution ---")
for v, c in sorted(ls["attack_column"]["distribution"].items(), key=lambda x: -x[1]):
    total = sum(ls["attack_column"]["distribution"].values())
    print(f"  {v:20s}: {c:>9,}  ({c/total*100:.2f}%)")

In [ ]:
# Visualise label imbalance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Binary label
binary_data = ls["label_column"]["distribution"]
axes[0].bar(binary_data.keys(), binary_data.values(), color=["#4CAF50", "#F44336"])
axes[0].set_title("Label Distribution (Binary)")
axes[0].set_xlabel("Label Value")
axes[0].set_ylabel("Count")
for i, (k, v) in enumerate(binary_data.items()):
    axes[0].text(i, v * 1.01, f"{v:,}", ha="center", fontsize=9)

# Attack categories
attack_data = ls["attack_column"]["distribution"]
sorted_attacks = sorted(attack_data.items(), key=lambda x: -x[1])
names, counts = zip(*sorted_attacks)
colors = ["#4CAF50" if n == "Benign" else "#F44336" for n in names]
axes[1].barh(names, counts, color=colors)
axes[1].set_title("Attack Category Distribution")
axes[1].set_xlabel("Count")
axes[1].invert_yaxis()
for i, c in enumerate(counts):
    axes[1].text(c * 1.01, i, f"{c:,}", va="center", fontsize=8)

plt.tight_layout()
figs_dir = root / "outputs" / "figures"
figs_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(figs_dir / "label_distribution.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved to outputs/figures/label_distribution.png")

## 6. High-Correlation Feature Pairs

In [ ]:
corr = report["correlations"]
print(f"Threshold: |r| >= {corr['correlation_threshold']}")
print(f"Sample size used: {corr['sample_size_used']:,}")
print(f"Pairs found: {corr['num_high_correlation_pairs']}
")
for p in corr["high_correlation_pairs"]:
    print(f"  r={p['pearson_r']:+.6f}  {p['feature_a']} <-> {p['feature_b']}")

## 7. Extreme-Value Features

In [ ]:
ev = report["extreme_values"]
print(f"Features exceeding mean + 5*std: {ev['num_extreme_features']}
")
for feat, s in ev["extreme_value_features"].items():
    print(f"  {feat:40s}  max={s['max']:>15,.2f}  mean={s['mean']:>12,.4f}  ratio={s['max_to_mean_ratio']}")

## 8. Temporal Provenance

In [ ]:
tp = report["temporal_provenance"]
print(f"Explicit time columns found  : {tp['explicit_temporal_columns_found']}")
print(f"Partial name matches         : {tp.get('partial_name_matches', [])}")
print(f"Temporal ordering established: {tp['temporal_ordering_established']}")
print(f"
Note:
{tp['notes']}")

## 9. Implications for GRU Sequential Model

| Finding | Impact |
|---|---|
| **No timestamp column present** | Row order cannot be treated as chronological without external evidence |
| **421 MB, 2.39M rows** | GRU windows must be formed carefully; full-dataset sliding window will be memory-intensive |
| **96% class imbalance (Label=0)** | GRU must use weighted loss / oversampling; raw accuracy is misleading |
| **25 extreme-value features** | Robust scaling (e.g. RobustScaler) preferred over StandardScaler for GRU inputs |
| **15 high-correlation pairs** | Feature redundancy exists; may reduce GRU input dimensionality in Stage 2 |
| **Attack has 9 attack classes** | GRU can be trained as binary (Label) or 10-class (Attack) — decision needed |



## 10. Outstanding Methodological Decisions Before Stage 2

1. **Temporal ordering strategy**: With no timestamp column, what defines a "sequence" for the GRU? Options:
   - Group by  and treat per-flow sub-sequences as windows.
   - Use raw row order with an explicit caveat that chronological ordering is unverified.
2. **GRU target**: Binary () or multiclass () — or both sequentially?
3. **Highly correlated feature pairs**: Drop one from each pair, or keep all 39 and let the model decide?
4. **Imbalance handling**: Weighted cross-entropy in GRU, or resample partitions during Dirichlet split?
5. **Extreme values**: RobustScaler vs MinMaxScaler vs log1p transform — decide before Stage 2 preprocessing.


## 11. Full Summary

In [ ]:
print_audit_summary(report)